<a href="https://colab.research.google.com/github/MiguelFigueroa02/ITAM_Sport_Lucho_FC/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import json
import pandas as pd
from pathlib import Path
import re

## 1. Carga de datos
---
A partir de los archivos json disponibles, procesamos los datos para construir un dataset que contenga todos los partidos del américa disponibles en el periodo 2023 - 2026

In [ ]:
DATA_DIR = Path("/content/")
OUTPUT_FILE = Path("/content/america_matches.csv")
AMERICA_ID = 1229

In [ ]:
json_files = sorted(DATA_DIR.glob("matches-*.json"))

In [ ]:
if not json_files:
    raise FileNotFoundError(
        f"No se encontraron archivos JSON en: {DATA_DIR.resolve()}"
    )

In [ ]:
records = []

for file in json_files:

    print(f"\nArchivo: {file.name}")

    with open(file, "r", encoding="utf-8") as f:
        matches = json.load(f)

    print(f"Partidos encontrados en archivo: {len(matches)}")

    for match in matches:

        home = match["home_team"]
        away = match["away_team"]

        home_id = home["home_team_id"]
        away_id = away["away_team_id"]

        if home_id != AMERICA_ID and away_id != AMERICA_ID:
            continue

        is_home = home_id == AMERICA_ID

        opponent = (
            away["away_team_name"]
            if is_home
            else home["home_team_name"]
        )

        america_score = (
            match["home_score"]
            if is_home
            else match["away_score"]
        )

        opponent_score = (
            match["away_score"]
            if is_home
            else match["home_score"]
        )


        managers = home.get("managers", []) if is_home else away.get("managers", [])

        coach = (
            managers[0]["name"]
            if managers
            else None
        )


        records.append({

            "match_id": match["match_id"],

            "match_date": match["match_date"],

            "kick_off": match.get("kick_off"),

            "season": match["season"]["season_name"],

            "competition": match["competition"]["competition_name"],

            "competition_stage": (
                match.get("competition_stage", {}).get("name")
            ),

            "match_week": match.get("match_week"),

            "opponent": opponent,

            "venue": "Home" if is_home else "Away",

            "america_score": america_score,

            "opponent_score": opponent_score,

            "coach": coach,

            "stadium": (
                match.get("stadium", {}).get("name")
            ),

            "match_status": match.get("match_status"),

            "match_status_360": match.get("match_status_360"),

        })



america_matches = pd.DataFrame(records)


if america_matches.empty:
    raise ValueError(
        "No se encontraron partidos del América. "
        "Revisa el AMERICA_ID."
    )


def get_result(row):

    if row["america_score"] > row["opponent_score"]:
        return "W"

    elif row["america_score"] == row["opponent_score"]:
        return "D"

    else:
        return "L"


america_matches["result"] = america_matches.apply(
    get_result,
    axis=1
)


america_matches["match_date"] = pd.to_datetime(
    america_matches["match_date"]
)


america_matches = america_matches.sort_values(
    ["match_date", "match_id"]
).reset_index(drop=True)



OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)



america_matches.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 60)
print("CATÁLOGO DEL AMÉRICA")
print("=" * 60)

print(f"\nTotal de partidos: {len(america_matches)}")

print("\nPartidos por temporada:")
print(
    america_matches
    .groupby("season")
    .size()
    .sort_index()
)

print("\nResultados:")
print(
    america_matches["result"]
    .value_counts()
)

print("\nLocal / Visitante:")
print(
    america_matches["venue"]
    .value_counts()
)

print("\nDisponibilidad de 360:")
print(
    america_matches["match_status_360"]
    .value_counts(dropna=False)
)

print("\nEntrenadores:")
print(
    america_matches["coach"]
    .value_counts(dropna=False)
)

print(f"\nArchivo generado:")
print(OUTPUT_FILE.resolve())

print("\nPrimeros registros:")
print(
    america_matches.head(10).to_string(index=False)
)


Archivo: matches-2023-2024.json
Partidos encontrados en archivo: 340

Archivo: matches-2024-2025.json
Partidos encontrados en archivo: 340

Archivo: matches-2025-2026.json
Partidos encontrados en archivo: 337

CATÁLOGO DEL AMÉRICA

Total de partidos: 131

Partidos por temporada:
season
2023/2024    46
2024/2025    47
2025/2026    38
dtype: int64

Resultados:
result
W    69
D    37
L    25
Name: count, dtype: int64

Local / Visitante:
venue
Away    66
Home    65
Name: count, dtype: int64

Disponibilidad de 360:
match_status_360
available      130
unscheduled      1
Name: count, dtype: int64

Entrenadores:
coach
André Soares Jardine              129
Diego Alberto Cervantes Chávez      2
Name: count, dtype: int64

Archivo generado:
/content/america_matches.csv

Primeros registros:
 match_id match_date     kick_off    season competition competition_stage  match_week    opponent venue  america_score  opponent_score                coach                stadium match_status match_status_360 r

## 2.  Auditoría inicial de los datos de los eventos de partidos.
---

Hacemos en primera instancia una evaluación inicial de los datos.

## 3. Integración y exploración de los datos

In [ ]:
def evaluacion(archivo):
  repo = 'https://raw.githubusercontent.com/MiguelFigueroa02/ITAM_Sport_Lucho_FC/refs/heads/main/data/raw/'
  path = repo + archivo
  CHUNK_SIZE = 50_000
  print("### Evaluación de eventos")
  header = pd.read_csv(
      path,
      nrows=0
    )
  columns = header.columns.tolist()
  print(f"\nNúmero de columnas: {len(columns)}")
  print("\nColumnas:")
  for i, col in enumerate(columns, start=1):
      print(f"{i:3}. {col}")

  total_rows = 0
  unique_matches = set()

  sample = None

  for chunk in pd.read_csv(
      path,
      chunksize=CHUNK_SIZE,
      low_memory=False,
      on_bad_lines='skip'
  ):

      total_rows += len(chunk)

      if sample is None:
          sample = chunk.head(5).copy()

      if "match_id" in chunk.columns:
          unique_matches.update(
              chunk["match_id"].dropna().unique()
          )

  print("### RESUMEN")

  print(f"\nFilas totales: {total_rows:,}")

  print(f"Partidos únicos: {len(unique_matches):,}")

  print("### MUESTRA DE EVENTOS")

  print(
    sample.to_csv(index=False)
  )
  print("### TIPOS DE DATOS")

  print(sample.dtypes)


In [ ]:
evaluacion('Liga_MX_2023_2024_America_p1.csv')

### Evaluación de eventos

Número de columnas: 251

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [ ]:
evaluacion('Liga_MX_2023_2024_America_p2.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [ ]:
evaluacion('Liga_MX_2023_2024_America_p3.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [ ]:
evaluacion('Liga_MX_2024_2025_America_p1.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [ ]:
evaluacion('Liga_MX_2024_2025_America_p2.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [ ]:
evaluacion('Liga_MX_2024_2025_America_p3.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [ ]:
evaluacion('Liga_MX_2025_2026_America_p1.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [ ]:
evaluacion('Liga_MX_2025_2026_America_p2.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

### Construcción de archivo RAW

In [5]:
repo = 'https://raw.githubusercontent.com/MiguelFigueroa02/ITAM_Sport_Lucho_FC/refs/heads/main/data/raw/'

In [8]:
datasets = [
    "Liga_MX_2023_2024_America_p1.csv",
    "Liga_MX_2023_2024_America_p2.csv",
    "Liga_MX_2023_2024_America_p3.csv",
    "Liga_MX_2024_2025_America_p1.csv",
    "Liga_MX_2024_2025_America_p2.csv",
    "Liga_MX_2024_2025_America_p3.csv",
    "Liga_MX_2025_2026_America_p1.csv",
    "Liga_MX_2025_2026_America_p2.csv"
]

In [11]:
for dataset in datasets:
    file = repo + dataset
    df = pd.read_csv(file, nrows=5)

    print(f"{dataset}")
    print(f"  Filas inspeccionadas: {len(df)}")
    print(f"  Columnas: {len(df.columns)}")
    # print(f"  Tamaño: {file.stat().st_size / 1024**2:.2f} MB")
    print()

Liga_MX_2023_2024_America_p1.csv
  Filas inspeccionadas: 5
  Columnas: 251

Liga_MX_2023_2024_America_p2.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2023_2024_America_p3.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2024_2025_America_p1.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2024_2025_America_p2.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2024_2025_America_p3.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2025_2026_America_p1.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2025_2026_America_p2.csv
  Filas inspeccionadas: 5
  Columnas: 254



In [13]:
schemas = {}

for dataset in datasets:
    file = repo + dataset
    df = pd.read_csv(file, nrows=0)
    schemas[dataset] = set(df.columns)

all_columns = set().union(*schemas.values())

print(f"Columnas totales distintas: {len(all_columns)}\n")

for filename, columns in schemas.items():
    missing = all_columns - columns
    extra = columns - min(schemas.values(), key=len)

    print(f"=== {filename} ===")
    print(f"Columnas: {len(columns)}")
    print(f"Faltantes respecto al universo: {len(missing)}")

    if missing:
        print("Faltantes:", sorted(missing))

    print()

Columnas totales distintas: 707

=== Liga_MX_2023_2024_America_p1.csv ===
Columnas: 251
Faltantes respecto al universo: 456
Faltantes: ['107179_defensive_responsibility_probability', '107179_interception_like_defensive_responsibility_probability', '107179_pressure_like_defensive_responsibility_probability', '10846_defensive_responsibility_probability', '10846_interception_like_defensive_responsibility_probability', '10846_pressure_like_defensive_responsibility_probability', '123690_defensive_responsibility_probability', '123690_interception_like_defensive_responsibility_probability', '123690_pressure_like_defensive_responsibility_probability', '130953_defensive_responsibility_probability', '130953_interception_like_defensive_responsibility_probability', '130953_pressure_like_defensive_responsibility_probability', '131123_defensive_responsibility_probability', '131123_interception_like_defensive_responsibility_probability', '131123_pressure_like_defensive_responsibility_probability', '1

In [16]:
# Tomamos como referencia uno de los archivos de 254 columnas
reference = datasets[0]
reference_file = repo + reference

reference_columns = set(
    pd.read_csv(reference_file, nrows=0).columns
)

for dataset in datasets[1:]:
    file = repo + dataset
    columns = set(pd.read_csv(file, nrows=0).columns)

    missing = reference_columns - columns
    extra = columns - reference_columns

    print(f'## {dataset}')
    print(f"Columnas: {len(columns)}")

    print(f"\nFaltantes respecto a {reference}: {len(missing)}")
    if missing:
        print(sorted(missing))

    print(f"\nExtra respecto a {reference}: {len(extra)}")
    if extra:
        print(sorted(extra))

## Liga_MX_2023_2024_America_p2.csv
Columnas: 254

Faltantes respecto a Liga_MX_2023_2024_America_p1.csv: 90
['11388_defensive_responsibility_probability', '11388_interception_like_defensive_responsibility_probability', '11388_pressure_like_defensive_responsibility_probability', '13034_defensive_responsibility_probability', '13034_interception_like_defensive_responsibility_probability', '13034_pressure_like_defensive_responsibility_probability', '131162_defensive_responsibility_probability', '131162_interception_like_defensive_responsibility_probability', '131162_pressure_like_defensive_responsibility_probability', '134157_defensive_responsibility_probability', '134157_interception_like_defensive_responsibility_probability', '134157_pressure_like_defensive_responsibility_probability', '182089_defensive_responsibility_probability', '182089_interception_like_defensive_responsibility_probability', '182089_pressure_like_defensive_responsibility_probability', '23842_defensive_responsibility

#### Interpretación
---
- Estamos encontrando que cada uno de los datasets presenta columnas dinámicas asociadas al ID de los jugadores de cada partido.
- Para continuar con nuestro proceso, procederemos a definir cuáles columnas son estables y cuáles son asociados a resultados dinámicos. A partir de las columnas estables son de las que en primera instancia procederemos a establecer métricas básicas asociadas al rendimiento para interpretación del estilo táctico.


In [18]:
columns = pd.read_csv(reference_file, nrows=0).columns

dynamic_columns = [
    col for col in columns
    if re.match(r"^\d+_", col)
]

stable_columns = [
    col for col in columns
    if col not in dynamic_columns
]

print(f"Columnas totales: {len(columns)}")
print(f"Columnas dinámicas de jugador: {len(dynamic_columns)}")
print(f"Columnas estables: {len(stable_columns)}")

print("\nPrimeras columnas estables:")
print(stable_columns[:50])

Columnas totales: 251
Columnas dinámicas de jugador: 123
Columnas estables: 128

Primeras columnas estables:
['id', 'index', 'match_id', 'period', 'timestamp', 'minute', 'second', 'event_type_id', 'event_type_name', 'possession', 'possession_team_id', 'possession_team_name', 'play_pattern_id', 'play_pattern_name', 'team_id', 'team_name', 'player_id', 'player_name', 'player_position_id', 'player_position_name', 'location_x', 'location_y', 'end_location_x', 'end_location_y', 'end_location_z', 'impact_height', 'duration', 'under_pressure', 'counterpress', 'substituted_player_id', 'substituted_player_name', 'pass_recipient_id', 'pass_recipient_name', 'pass_length', 'pass_angle', 'pass_height_id', 'pass_height_name', 'pass_cross', 'pass_cut_back', 'pass_switch', 'assisted_shot_id', 'key_pass_id', 'follows_dribble', 'one_on_one', 'open_goal', 'statsbomb_xg', 'off_camera', 'out', 'obv_for_after', 'obv_for_before']


In [20]:
# Comparar las columnas estables entre todos los archivos

reference_columns = set(stable_columns)

for dataset in datasets:
    file = repo + dataset
    columns = set(pd.read_csv(file, nrows=0).columns)

    dynamic = {
        col for col in columns
        if re.match(r"^\d+_", col)
    }

    stable = columns - dynamic

    missing = reference_columns - stable
    extra = stable - reference_columns

    print(f"\n{'='*70}")
    print(dataset)
    print(f"Columnas estables: {len(stable)}")
    print(f"Faltantes: {len(missing)}")
    print(f"Extras: {len(extra)}")

    if missing:
        print("Faltantes:", sorted(missing))

    if extra:
        print("Extras:", sorted(extra))


Liga_MX_2023_2024_America_p1.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2023_2024_America_p2.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2023_2024_America_p3.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2024_2025_America_p1.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2024_2025_America_p2.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2024_2025_America_p3.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2025_2026_America_p1.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2025_2026_America_p2.csv
Columnas estables: 128
Faltantes: 0
Extras: 0


#### Concatenación
---
A partir de las columnas estables detectadas, procederemos a hacer la configuración de un dataset maestro


In [21]:
stable_columns = list(stable_columns)

dfs = []

for dataset in datasets:
    file = repo + dataset
    print(f"Cargando: {dataset}")

    df = pd.read_csv(file, usecols=stable_columns)

    # Trazabilidad
    df["source_file"] = dataset

    dfs.append(df)


# Concatenar los 8 archivos
events_raw = pd.concat(
    dfs,
    ignore_index=True
)

print("## DATASET CONSOLIDADO")

print(f"Filas: {len(events_raw):,}")
print(f"Columnas: {len(events_raw.columns)}")
print(f"Partidos únicos: {events_raw['match_id'].nunique()}")

print("\nPartidos por archivo:")
print(
    events_raw.groupby("source_file")["match_id"]
    .nunique()
)

print("\nMemoria utilizada:")
print(f"{events_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Cargando: Liga_MX_2023_2024_America_p1.csv


/tmp/ipykernel_1640/898783433.py:9: DtypeWarning: Columns (60,97,99,246,247,249) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, usecols=stable_columns)


Cargando: Liga_MX_2023_2024_America_p2.csv


/tmp/ipykernel_1640/898783433.py:9: DtypeWarning: Columns (60,90,92,93,97,99) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, usecols=stable_columns)


Cargando: Liga_MX_2023_2024_America_p3.csv


...

Cargando: Liga_MX_2024_2025_America_p2.csv
Cargando: Liga_MX_2024_2025_America_p3.csv


/tmp/ipykernel_1640/898783433.py:9: DtypeWarning: Columns (17,19,30,32,36,37,38,39,40,41,43,44,56,58,60,62,64,66,88,90,92,93,101,102,104,249,250,253) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, usecols=stable_columns)


Cargando: Liga_MX_2025_2026_America_p1.csv


/tmp/ipykernel_1640/898783433.py:9: DtypeWarning: Columns (249,253) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, usecols=stable_columns)


Cargando: Liga_MX_2025_2026_America_p2.csv
## DATASET CONSOLIDADO
Filas: 501,944
Columnas: 129
Partidos únicos: 131

Partidos por archivo:
source_file
Liga_MX_2023_2024_America_p1.csv    20
Liga_MX_2023_2024_America_p2.csv    20
Liga_MX_2023_2024_America_p3.csv     6
Liga_MX_2024_2025_America_p1.csv    20
Liga_MX_2024_2025_America_p2.csv    20
Liga_MX_2024_2025_America_p3.csv     7
Liga_MX_2025_2026_America_p1.csv    20
Liga_MX_2025_2026_America_p2.csv    18
Name: match_id, dtype: int64

Memoria utilizada:
1119.55 MB


Comparación de datos generados a partir del dataset procesado de partidos

In [23]:
partidos = 'https://raw.githubusercontent.com/MiguelFigueroa02/ITAM_Sport_Lucho_FC/refs/heads/main/data/processed/america_matches.csv'
matches = pd.read_csv(partidos)

print("Dimensiones:", matches.shape)

print("\nColumnas:")
print(matches.columns.tolist())

print("\nPrimeras filas:")
display(matches.head().to_markdown())

print("\nPartidos únicos:")
print(matches["match_id"].nunique())

print("\nDuplicados de match_id:")
print(matches["match_id"].duplicated().sum())

print("\nTipos de datos:")
print(matches.dtypes)

Dimensiones: (131, 16)

Columnas:
['match_id', 'match_date', 'kick_off', 'season', 'competition', 'competition_stage', 'match_week', 'opponent', 'venue', 'america_score', 'opponent_score', 'coach', 'stadium', 'match_status', 'match_status_360', 'result']

Primeras filas:


'|    |   match_id | match_date   | kick_off     | season    | competition   | competition_stage   |   match_week | opponent   | venue   |   america_score |   opponent_score | coach                | stadium         | match_status   | match_status_360   | result   |\n|---:|-----------:|:-------------|:-------------|:----------|:--------------|:--------------------|-------------:|:-----------|:--------|----------------:|-----------------:|:---------------------|:----------------|:---------------|:-------------------|:---------|\n|  0 |    3889637 | 2023-07-01   | 01:00:00.000 | 2023/2024 | Liga MX       | Apertura            |            1 | Juárez     | Home    |               1 |                2 | André Soares Jardine | Estadio Azteca  | available      | available          | L        |\n|  1 |    3889660 | 2023-07-16   | 01:00:00.000 | 2023/2024 | Liga MX       | Apertura            |            3 | Puebla     | Home    |               3 |                0 | André Soares Jardine | Est


Partidos únicos:
131

Duplicados de match_id:
0

Tipos de datos:
match_id              int64
match_date           object
kick_off             object
season               object
competition          object
competition_stage    object
match_week            int64
opponent             object
venue                object
america_score         int64
opponent_score        int64
coach                object
stadium              object
match_status         object
match_status_360     object
result               object
dtype: object


In [24]:
event_match_ids = set(events_raw["match_id"].unique())
match_match_ids = set(matches["match_id"].unique())

print("Partidos en eventos:", len(event_match_ids))
print("Partidos en catálogo:", len(match_match_ids))

print("\nEventos sin metadata:")
print(event_match_ids - match_match_ids)

print("\nPartidos del catálogo sin eventos:")
print(match_match_ids - event_match_ids)

Partidos en eventos: 131
Partidos en catálogo: 131

Eventos sin metadata:
set()

Partidos del catálogo sin eventos:
set()


Una vez que comprobamos que los partidos de ambos datasets coinciden, procederemos a hacer un join para generar nuestro dataset maestro

In [25]:
events = events_raw.merge(
    matches,
    on="match_id",
    how="left",
    validate="many_to_one"
)

print("Dimensiones:", events.shape)

print("\nPartidos únicos:", events["match_id"].nunique())

print("\nColumnas de contexto agregadas:")
print([
    "season",
    "match_date",
    "opponent",
    "venue",
    "america_score",
    "opponent_score",
    "coach",
    "result"
])

print("\nValores nulos en contexto:")
print(
    events[
        [
            "season",
            "match_date",
            "opponent",
            "venue",
            "america_score",
            "opponent_score",
            "coach",
            "result"
        ]
    ].isna().sum()
)

Dimensiones: (501944, 144)

Partidos únicos: 131

Columnas de contexto agregadas:
['season', 'match_date', 'opponent', 'venue', 'america_score', 'opponent_score', 'coach', 'result']

Valores nulos en contexto:
season            0
match_date        0
opponent          0
venue             0
america_score     0
opponent_score    0
coach             0
result            0
dtype: int64


Queremos conocer el tipo de eventos asociados a nuestro dataset generado

In [26]:
print("EVENT TYPES")
print(events["event_type_name"].value_counts())

print("\n" + "="*70)
print("PLAY PATTERNS")
print(events["play_pattern_name"].value_counts())

print("\n" + "="*70)
print("POSSESSION TEAM")
print(events["possession_team_name"].value_counts().head(10))

print("\n" + "="*70)
print("RESULTADOS")
print(events["result"].value_counts())

print("\n" + "="*70)
print("LOCALÍA")
print(events["venue"].value_counts())

EVENT TYPES
event_type_name
Pass                 126618
Ball Receipt*        116392
Carries              103365
Shot                  47888
Pressure              41287
Ball Recovery         11619
Duel                   7793
Tactical Shift         5734
Clearance              4611
Block                  4542
Goal Keeper            3974
Miscontrol             3475
Dribble                3321
Foul Committed         3269
Starting XI            3144
Foul Won               3084
Dispossessed           3078
Dribbled Past          1901
Interception           1878
Substitution           1177
Injury Stoppage         824
50/50                   665
Half Start              530
Half End                530
Referee Ball-Drop       455
Player Off              186
Player On               185
Shield                  170
Bad Behaviour           134
Error                    62
Offside                  43
Own Goal For              5
Own Goal Against          5
Name: count, dtype: int64

PLAY PATTERNS
play_pa

## 4. Construcción

### Pases por posesión


In [28]:
TEAM_ID = 1229

In [ ]:
events.possession_team_name

In [42]:
def possesions(team_id):
  team_events = events[
      events["possession_team_id"] == team_id
  ].copy()
  team_name = team_events.possession_team_name.unique()[0]

  print(f"Eventos de {team_name}:", len(team_events))
  print("Posesiones:", team_events["possession"].nunique())

  print("\nPrimeras posesiones:")
  print(
      team_events[
          ["match_id", "possession", "event_type_name",
          "player_name", "minute"]
      ].head(20).to_markdown()
  )

  print("\nEvento por posesiones:")
  possession_size = (
      team_events.groupby(["match_id", "possession"]).size().reset_index(name="events_in_possession")
  )

  print(possession_size.head().to_markdown())

  print("\nEstadísticas:")
  print(
      possession_size["events_in_possession"].describe().to_markdown()
  )

In [43]:
possesions(TEAM_ID)

Eventos de América: 272145
Posesiones: 249

Primeras posesiones:
|    |   match_id |   possession | event_type_name   | player_name                     |   minute |
|---:|-----------:|-------------:|:------------------|:--------------------------------|---------:|
|  4 |    3889637 |            2 | Pass              | Román Martínez                  |        0 |
|  5 |    3889637 |            2 | Ball Receipt*     | Jonathan dos Santos Ramírez     |        0 |
|  6 |    3889637 |            2 | Pass              | Jonathan dos Santos Ramírez     |        0 |
|  7 |    3889637 |            2 | Ball Receipt*     | Richard Rafael Sánchez Guerrero |        0 |
|  8 |    3889637 |            2 | Pass              | Richard Rafael Sánchez Guerrero |        0 |
|  9 |    3889637 |            2 | Ball Receipt*     | Kevin Nahin Álvarez Campos      |        0 |
| 10 |    3889637 |            2 | Carries           | Kevin Nahin Álvarez Campos      |        0 |
| 11 |    3889637 |            2 | 

###